# Анализ тональности твитов о коронавирусе
***В период пандемии COVID-19 социальные сети стали основным каналом для выражения эмоций. Анализ тональности (sentiment analysis) твитов в подобные кризисные периоды позволяет отслеживать общественные настроения и выявлять тревожные тенденции у населения.***

**Цель**: построить классификатор, который по тексту твита определяет его эмоциональную окраску (положительную или отрицательную). Для этого я использовала классические методы NLP: токенизацию, удаление стоп-слов, стемминг, векторизацию с помощью Bag-of-Words и TF-IDF, а также логистическую регрессию в качестве модели.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import nltk
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from nltk.tokenize import TweetTokenizer
from string import punctuation
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from scipy.sparse import hstack
import warnings
warnings.filterwarnings('ignore')

In [2]:
from src.tokenizers import custom_tokenizer, custom_stem_tokenizer
from src.model_eval import evaluate_model

# Предобработка данных

In [3]:
df = pd.read_csv('data/tweets_coronavirus.csv', encoding='latin-1')
df.head()

,UserName,ScreenName,Location,TweetAt,OriginalTweet,Sentiment
0,3800,48752,UK,16-03-2020,advice Talk to your neighbours family to excha...,Positive
1,3801,48753,Vagabonds,16-03-2020,Coronavirus Australia: Woolworths to give elde...,Positive
2,3802,48754,NaN,16-03-2020,My food stock is not the only one which is emp...,Positive
3,3803,48755,NaN,16-03-2020,"Me, ready to go at supermarket during the #COV...",Extremely Negative
4,3804,48756,"ÃÂT: 36.319708,-82.363649",16-03-2020,As news of the regionÃÂs first confirmed COV...,Positive


In [4]:
df['Binary_sentiment'] = df['Sentiment'].apply(lambda x: 1 if x in ['Positive', 'Extremely Positive'] else 0)

In [5]:
df['Location'] = df['Location'].fillna('Unknown')

In [6]:
X = df['OriginalTweet']
y = df['Binary_sentiment']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

# Векторизация текстов

## CountVectorizer

In [7]:
cv = CountVectorizer(tokenizer=custom_tokenizer)
cv.fit(X_train)
print(f'Размер словаря CountVectorizer: {len(cv.vocabulary_)}')

Размер словаря CountVectorizer: 45083


Рассмотрим какой-нибудь конкретный твит и определим в нем наиболее и наименее важные токены 

In [8]:
ind = 9023
tw_text = df.iloc[ind]['OriginalTweet']
tw_sentiment = df.iloc[ind]['Sentiment']
print(tw_text)
print(f'Sentiment: {tw_sentiment}')

Shop keepers taking advantage of #Coronavirus boosting prices disproportionately so the most Marginalised suffer who can't afford it #SHAMEONYOU #Wewillremember
Sentiment: Negative


In [9]:
tw_vector = cv.transform([tw_text])
feature_names = cv.get_feature_names_out()
tw_array = tw_vector.toarray()[0]
non_zero_ind = np.where(tw_array > 0)[0]

max_index = non_zero_ind[np.argmax(tw_array[non_zero_ind])]
min_index = non_zero_ind[np.argmin(tw_array[non_zero_ind])]
print(f"Наиболее важный токен: '{feature_names[max_index]}' — {tw_array[max_index]}")
print(f"Наименее важный токен: '{feature_names[min_index]}' — {tw_array[min_index]}")

Наиболее важный токен: '#coronavirus' — 1
Наименее важный токен: '#coronavirus' — 1


Cамый важный и самый неважный токены (компоненты которых в векторе максимальны/минимальны, без учета 0) совпадают, потому что каждый токен в твите встречается по одному разу (у всех токенов одинаковая частота 1), и argmax с argmin берут первый попавшийся токен. значит, **CountVectorizer плохо подходит для классификации**, так как определяет важность только по частоте в одном твите

## TfidfVectorizer

In [10]:
tfidf = TfidfVectorizer(tokenizer=custom_tokenizer)
tfidf.fit(X_train)
print(f'Размер словаря TfidfVectorizer: {len(tfidf.vocabulary_)}')

Размер словаря TfidfVectorizer: 45083


In [11]:
tw_vector_tfidf = tfidf.transform([tw_text])
feature_names_tfidf = tfidf.get_feature_names_out()
tweet_array_tfidf = tw_vector_tfidf.toarray()[0]
non_zero_tfidf = np.where(tweet_array_tfidf > 0)[0]
max_idx_tfidf = non_zero_tfidf[np.argmax(tweet_array_tfidf[non_zero_tfidf])]
min_idx_tfidf = non_zero_tfidf[np.argmin(tweet_array_tfidf[non_zero_tfidf])]
    
print(f"Наиболее важный токен: '{feature_names_tfidf[max_idx_tfidf]}' — {tweet_array_tfidf[max_idx_tfidf]:.4f}")
print(f"Наименее важный токен: '{feature_names_tfidf[min_idx_tfidf]}' — {tweet_array_tfidf[min_idx_tfidf]:.4f}")

Наиболее важный токен: '#wewillremember' — 0.3760
Наименее важный токен: '#coronavirus' — 0.0720


#wewillremember - редкий хештег во всем корпусе, поэтому получил высокий вес (0,376), а #coronavirus - очень частый хештег среди всех твитов, получил низкий вес (0,072). Можем сделать вывод, что **TF-IDF лучше подходит для классификации**, потому что выделяет редкие информативные токены и игнорирует общие слова

Проверим, как работает TfidfVectorizer, на каком-нибудь положительном твите

In [12]:
pos_ind = 43
pos_tweet = df.iloc[pos_ind]['OriginalTweet']
print(pos_tweet)
print(f'Sentiment: {df.iloc[pos_ind]['Sentiment']}')

Morning everyone have a great and safe day. ??? #coronavirus #StopPanicBuying #BeKind #mufc #MUFC_Family
Sentiment: Extremely Positive


In [13]:
tw_vector_tfidf = tfidf.transform([pos_tweet])
feature_names_tfidf = tfidf.get_feature_names_out()
tweet_array_tfidf = tw_vector_tfidf.toarray()[0]
non_zero_tfidf = np.where(tweet_array_tfidf > 0)[0]
max_idx_tfidf = non_zero_tfidf[np.argmax(tweet_array_tfidf[non_zero_tfidf])]
min_idx_tfidf = non_zero_tfidf[np.argmin(tweet_array_tfidf[non_zero_tfidf])]
    
print(f"Наиболее важный токен: '{feature_names_tfidf[max_idx_tfidf]}' — {tweet_array_tfidf[max_idx_tfidf]:.4f}")
print(f"Наименее важный токен: '{feature_names_tfidf[min_idx_tfidf]}' — {tweet_array_tfidf[min_idx_tfidf]:.4f}")

Наиболее важный токен: '#bekind' — 0.5091
Наименее важный токен: '#coronavirus' — 0.1346


На примере нашего резко положительного твита TfidfVectorizer хорошо выделяет важные токены. #bekind - редкий хештег, несет прямую положительную эмоциональную окраску, tfidf правильно определил ему большой вес (0,5091).

# Обучение моделей без стемминга

## Наивный классификатор
До обучения сложных моделей оценим, какую точность даёт самый простой подход — предсказание самого частого класса.

In [14]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)
dummy_accuracy = accuracy_score(y_test, y_pred_dummy)
print(f"Accuracy DummyClassifier: {dummy_accuracy:.4f}")

Accuracy DummyClassifier: 0.5396


## Логистическая регрессия

Сравним CountVectorizer и TfidfVectorizer, обучив логистическую регрессию на векторах из обоих векторайзеров

In [15]:
X_train_cv = cv.transform(X_train)
X_test_cv = cv.transform(X_test)

X_train_tfidf = tfidf.transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [16]:
results = []
metrics_cv = evaluate_model(LogisticRegression(max_iter=1000, random_state=0), X_train_cv, y_train, X_test_cv, y_test, model_name='CV')
metrics_tfidf = evaluate_model(LogisticRegression(max_iter=1000, random_state=0), X_train_tfidf, y_train, X_test_tfidf, y_test, model_name='Tfidf')
results.append(metrics_cv)
results.append(metrics_tfidf)

In [17]:
res_df = pd.DataFrame(results).round(4)
res_df

,Model,Train Accuracy,Test Accuracy,Train F1-Score,Test F1-Score,Train ROC-AUC,Test ROC-AUC
0,CV,0.9837,0.8731,0.9849,0.8832,0.9974,0.9402
1,Tfidf,0.9251,0.8513,0.9315,0.8646,0.9733,0.9292


Обе модели показали высокие результаты на тесте (accuracy > 85%). CV показал более высокую точность на тесте, но при этом большое переобучение (разрыв между train и test 11%), а TF-IDF показал меньшее переобучение, но ниже точность на тесте. То есть TF-IDF будет лучше работать на новых данных

# Улучшение токенизации: стемминг

Используем модифицированную версию собственного токенайзера, в которую входит стемминг

In [18]:
cv_stem = CountVectorizer(tokenizer=custom_stem_tokenizer)
X_train_cv_stem = cv_stem.fit_transform(X_train)

print(f'Размер словаря после стемминга: {len(cv_stem.vocabulary_)}')

Размер словаря после стемминга: 36369


Размер словаря уменьшился с 45083 до 36369, потому что при стемминге у слов отбрасываются суффиксы и окончания, что помогает сократить количество уникальных токенов без потери смысла

## Обучение моделей со стеммингом

In [19]:
X_test_cv_stem = cv_stem.transform(X_test)

tfidf_stem = TfidfVectorizer(tokenizer=custom_stem_tokenizer)
X_train_tfidf_stem = tfidf_stem.fit_transform(X_train)
X_test_tfidf_stem = tfidf_stem.transform(X_test)

In [20]:
metrics_cv_stem = evaluate_model(LogisticRegression(max_iter=1000, random_state=0), X_train_cv_stem, y_train, X_test_cv_stem, y_test, model_name='CV stemming')
metrics_tfidf_stem = evaluate_model(LogisticRegression(max_iter=1000, random_state=0), X_train_tfidf_stem, y_train, X_test_tfidf_stem, y_test, model_name='Tfidf stemming')
results.append(metrics_cv_stem)
results.append(metrics_tfidf_stem)

In [21]:
stem_df = pd.DataFrame(results).round(4)
stem_df

,Model,Train Accuracy,Test Accuracy,Train F1-Score,Test F1-Score,Train ROC-AUC,Test ROC-AUC
0,CV,0.9837,0.8731,0.9849,0.8832,0.9974,0.9402
1,Tfidf,0.9251,0.8513,0.9315,0.8646,0.9733,0.9292
2,CV stemming,0.9707,0.8708,0.9729,0.8813,0.9936,0.9376
3,Tfidf stemming,0.9170,0.8599,0.9240,0.8722,0.9697,0.9315


**Ответ:** результаты примерно такие же, что и без стемминга (почти везде чуть ниже точность). но все равно есть смысл использовать стемминг для экономии памяти (меньше размер словаря), а также потому что с ним разрыв между train и test уменьшился для обеих моделей (снизилось переобучение), и в случае TF-IDF немного выросла точность на тесте (без стемминга 0,8518, со стемингом 0,8593). 

пока самая лучшая модель - TF-IDF со стеммингом, так как показывает наименьшее переобучение и самую высокую точноть среди TF-IDF на тесте (более стабильная и обобщаемая)